Импорт библиотек

In [ ]:
import os
import mlflow

from sklearn.model_selection import train_test_split
import pandas as pd
import numpy

from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error

Загрузить очищенную выборку. Разбить ее на тестовую и обучающую в пропорции 25%-75%

In [ ]:
from pathlib import Path

def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "data" / "clean_data.pkl").exists() or (candidate / "data" / "dataset.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Не найден data/clean_data.pkl. Сначала выполните eda/eda.ipynb "
        f"(текущая директория: {here})"
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"

df = pd.read_pickle(DATA_DIR / "clean_data.pkl")
df = df.rename(columns={'Selling_Price': 'target'})
X_train, X_test, y_train, y_test = train_test_split(df.drop('target', axis=1), df['target'], test_size=0.25, random_state=2)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("train/test:", X_train.shape, X_test.shape)


Создать переменные, содержащие названия столбцов с числовыми и категориальными признаками

In [ ]:
cat_features = X_train.select_dtypes(include=['category','object']).columns.to_list()
num_features = X_train.select_dtypes(include=['number']).columns.to_list()

Создать pipeline обработки признаков и обучения модели.

In [ ]:
s_scaler = StandardScaler()
l_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=99999999) # unknown_value нужно выбирать с умом
regressor = RandomForestRegressor()

Обучить baseline-модель и получить метрики качества на тестовой выборке

In [ ]:
# Для удобной работы со столбцами
preprocessor = ColumnTransformer(
    transformers=[
        ('num', s_scaler, num_features),  # преобразования для числовых признаков
        ('cat', l_encoder, cat_features), # преобразования для категориальных признаков
    ],
    remainder='drop' ) # Удаляем столбцы, которые не затронуты преобразования

In [ ]:
pipeline = Pipeline(steps=[('preprocessor', preprocessor),  ('model', regressor)])

pipeline.fit(X_train, y_train)

In [ ]:
predictions = pipeline.predict(X_test) 

metrics = {}
metrics["mae"] = mean_absolute_error(y_test, predictions)   
metrics["mape"] = mean_absolute_percentage_error(y_test, predictions)
metrics["mse"] = mean_squared_error(y_test, predictions)

metrics

    Залогировать baseline-модель в новый эксперимент. Зарегистрировать модель.

In [ ]:
# Работаем с MLflow локально
TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5001

registry_uri = f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}"
tracking_uri = f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}"

mlflow.set_tracking_uri(tracking_uri)   
mlflow.set_registry_uri(registry_uri)  

In [ ]:
# название тестового эксперимента, запуска (run) внутри него, имени, под которым модель будет регистрироваться
EXPERIMENT_NAME = "estate_project"
RUN_NAME = "baseline model"
REGISTRY_MODEL_NAME = "estate_model_rf"

In [ ]:
from mlflow.models import infer_signature
input_example = X_train.head(5)
signature =  infer_signature(model_input = X_train.head(5))

In [ ]:
# Будем логировать requirements и артефакт - текстовый файл
req_file = '../requirements.txt'
art = '../comment.txt'
params_dict = pipeline.get_params()


In [ ]:
# Когда создаем новый эксперимент, то: 
exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
experiment_id = exp.experiment_id if exp is not None else mlflow.create_experiment(EXPERIMENT_NAME)

# Впоследствии. чтобы добавлять запуски в этот же эксепримент мы должны получить его id:
#experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    # получаем уникальный идентификатор запуска эксперимента
    run_id = run.info.run_id 
    mlflow.sklearn.log_model(pipeline, 
                             artifact_path="models",
                             signature=signature,
                             input_example=input_example,
                             pip_requirements=req_file
                             )
    mlflow.log_metrics(metrics)
    mlflow.log_artifact(art)
    mlflow.log_params(params_dict)

run = mlflow.get_run(run_id) 
assert (run.info.status =='FINISHED')

С использованием библиотеки sklearn создать дополнительные признаки, обучить модель, залогировать ее.

In [ ]:
from sklearn.preprocessing import QuantileTransformer, SplineTransformer, PolynomialFeatures, MinMaxScaler

In [ ]:
X_train_sklearn = X_train.copy()

In [ ]:
pf = PolynomialFeatures(degree=2)

In [ ]:
pf.fit_transform(X_train_sklearn[['Year','Driven_kms']])

In [ ]:
sp = SplineTransformer(n_knots=3, degree=3)

In [ ]:
sp.fit_transform(X_train_sklearn[['Driven_kms']])

In [ ]:
qt = QuantileTransformer(n_quantiles=100)

In [ ]:
qt.fit_transform(X_train_sklearn[['Driven_kms']])

In [ ]:
pf = PolynomialFeatures(degree=2)
qt = QuantileTransformer(n_quantiles=100)
sp = SplineTransformer(n_knots=3, degree=3)

In [ ]:
# Значения преобразованных признаков нужно отскейлить, поэтому создаем pipeline из двух шагов - преобразование и скейлинг
pf_pipeline = Pipeline(steps=[
    ('poly', pf),
    ('scale', StandardScaler())
])

In [ ]:
preprocessor_sklearn = ColumnTransformer(
    transformers=[
        ('num', s_scaler, num_features),  # преобразования для числовых признаков
        ('cat', l_encoder, cat_features), # преобразования для категориальных признаков
        ('quantile', qt,num_features),
        ('poly', pf_pipeline, ['Year', 'Driven_kms']), # В преобразования добавляем созданный ранее pipeline
        ('spline', sp, ['Driven_kms'])
    ],
    remainder='drop',
    ) # Удаляем столбцы, которые не затронуты преобразования

In [ ]:
X_train_sklearn_raw = preprocessor_sklearn.fit_transform(X_train_sklearn)
X_train_sklearn = pd.DataFrame(X_train_sklearn_raw, columns=preprocessor_sklearn.get_feature_names_out())

In [ ]:
# Удобно использовать для отображения всех строк\столбцов в DataFrame
with pd.option_context('display.max_rows', 5, 'display.max_columns', None):
    display (X_train_sklearn)

In [ ]:
pipeline_sklearn = Pipeline(steps=[
    ('transform', preprocessor_sklearn),
    ('model', regressor)
])

model_sklearn = pipeline_sklearn.fit(X_train, y_train)

In [ ]:
predictions = model_sklearn.predict(X_test) 
metrics = {}
metrics["mae"] = mean_absolute_error(y_test, predictions)   
metrics["mape"] = mean_absolute_percentage_error(y_test, predictions)
metrics["mse"] = mean_squared_error(y_test, predictions)

metrics

In [ ]:
experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
RUN_NAME = 'fe_sklearn'
art = '../comment.txt'
with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    # получаем уникальный идентификатор запуска эксперимента
    run_id = run.info.run_id 
    mlflow.sklearn.log_model(model_sklearn, 
                             artifact_path="models",
                             signature=signature,
                             input_example=input_example,
                             pip_requirements=req_file
                             )
    mlflow.log_metrics(metrics)
    mlflow.log_artifact(art)
    mlflow.log_params(model_sklearn.get_params())

run = mlflow.get_run(run_id) 
assert (run.info.status =='FINISHED')

Отобрать N наиболее важных признаков экспертно (по результатам EDA). N — ориентировочно 20–70% от числа признаков после FE. Обучить модель, залогировать ее.

In [ ]:
# Экспертный отбор по выводам EDA:
# Present_Price — главный предиктор; Year; Driven_kms / Driven_run;
# категориальные Fuel_Type, Selling_type, Transmission, Car_Name.
feature_names = list(X_train_sklearn.columns)
priority_keys = [
    'Present_Price', 'Year', 'Driven_kms', 'Driven_run',
    'Fuel_Type', 'Selling_type', 'Transmission', 'Car_Name',
]
preferred = [c for c in feature_names if any(k in c for k in priority_keys)]

# Берём ~40% признаков (в диапазоне 20–70%)
n_select = max(3, int(round(len(feature_names) * 0.4)))
if len(preferred) >= n_select:
    selected_cols = preferred[:n_select]
else:
    # если приоритетных меньше N — дополняем остальными
    rest = [c for c in feature_names if c not in preferred]
    selected_cols = preferred + rest[: n_select - len(preferred)]

selected_idx = [feature_names.index(c) for c in selected_cols]
print('Всего признаков после FE:', len(feature_names))
print('Отобрано N =', len(selected_cols))
print('Столбцы:', selected_cols)
print('Индексы:', selected_idx)


In [ ]:
selected_features = X_train_sklearn.loc[:, selected_cols]
selected_features.head()


In [ ]:
class ColumnExtractor(object):

    def __init__(self, cols):
        self.cols = cols

    def transform(self, X):
        return X[:, self.cols]

    def fit(self, X, y=None):
        return self


In [ ]:
with open('selected_skl_idx.txt', 'w+') as f:
    f.write(str(selected_idx))
with open('selected_skl_cols.txt', 'w+') as f:
    f.write(str(selected_cols))


In [ ]:
fs_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_sklearn),
    ('feature_selector', ColumnExtractor(selected_idx)),
    ('model', regressor),
])

fs_pipeline.fit(X_train, y_train)


In [ ]:
predictions_fs = fs_pipeline.predict(X_test)

metrics = {}
metrics["mae"] = mean_absolute_error(y_test, predictions_fs)
metrics["mape"] = mean_absolute_percentage_error(y_test, predictions_fs)
metrics["mse"] = mean_squared_error(y_test, predictions_fs)

experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
RUN_NAME = 'expert_feature_selection'

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    mlflow.sklearn.log_model(
        fs_pipeline,
        artifact_path="models",
        signature=signature,
        input_example=input_example,
        pip_requirements=req_file,
    )
    mlflow.log_metrics(metrics)
    mlflow.log_artifact('selected_skl_cols.txt')
    mlflow.log_artifact('selected_skl_idx.txt')
    mlflow.log_params(fs_pipeline.get_params())

run = mlflow.get_run(run_id)
assert run.info.status == 'FINISHED'
metrics


С помощью optuna настроить оптимальные параметры для модели, показывающей лучший результат. Обучить модель, залогировать ее, зарегистрировать очередную версию.

In [ ]:
import optuna

In [ ]:
def objective(trial):
    # предлагаем гиперпараметры
    depth = trial.suggest_int('depth', 1, 10)
    max_features = trial.suggest_float('max_features', 0.1, 1)
    n_estimators = trial.suggest_int('n_estimators', 1, 3)

    # создаём и обучаем модель
    opt_pipeline  = Pipeline(steps=[
        ('preprocessor', preprocessor_sklearn), 
        ('feature_selector', ColumnExtractor(selected_idx)),
        ('model',  RandomForestRegressor(n_estimators=n_estimators, max_depth=depth, max_features =max_features))
    ])

    opt_pipeline.fit(X_train, y_train)

    # предсказываем и вычисляем RMSE
    preds = opt_pipeline.predict(X_test)
    mae =  mean_absolute_error(y_test, preds)   

    return mae

In [ ]:
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=10)

# выводим результаты
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params) 

In [ ]:
opt_pipeline  = Pipeline(steps=[
    ('preprocessor', preprocessor_sklearn), 
    ('feature_selector', ColumnExtractor(selected_idx)),
    ('model', RandomForestRegressor(max_depth=8,max_features = 0.5052449805697352, n_estimators=3))
])

# Проведем стандартную проверку на тестовом множестве и залогируем run

In [ ]:
fs_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_sklearn), 
    ('feature_selector', ColumnExtractor(selected_idx)),
    ('model', regressor)
])

fs_pipeline.fit(X_train, y_train)

In [ ]:
predictions = fs_pipeline.predict(X_test) 
metrics = {}
metrics["mae"] = mean_absolute_error(y_test, predictions)   
metrics["mape"] = mean_absolute_percentage_error(y_test, predictions)
metrics["mse"] = mean_squared_error(y_test, predictions)

metrics

In [ ]:
experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
RUN_NAME = 'optuna'
art = '../comment.txt'
with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    # получаем уникальный идентификатор запуска эксперимента
    run_id = run.info.run_id 
    opt_pipeline.fit(X_train, y_train)
    predictions = opt_pipeline.predict(X_test)
    metrics = {}
    metrics["mae"] = mean_absolute_error(y_test, predictions)
    metrics["mape"] = mean_absolute_percentage_error(y_test, predictions)
    metrics["mse"] = mean_squared_error(y_test, predictions)
    mlflow.sklearn.log_model(opt_pipeline, 
                             artifact_path="models",
                             signature=signature,
                             input_example=input_example,
                             pip_requirements=req_file
                             )
    mlflow.log_metrics(metrics)
    mlflow.log_artifact(art)
    mlflow.log_params(opt_pipeline.get_params())

run = mlflow.get_run(run_id) 
assert (run.info.status =='FINISHED')

Проанализировать все прогоны и выбрать модель, показывающую наилучшее качество. Обучить эту модель на всей выборке (а не только на train-части), залогировать ее.

In [ ]:
from pathlib import Path

def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "data" / "clean_data.pkl").exists() or (candidate / "data" / "dataset.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Не найден data/clean_data.pkl. Сначала выполните eda/eda.ipynb "
        f"(текущая директория: {here})"
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"

# Обучаем финальную модель на ВСЕЙ выборке
df_full = pd.read_pickle(DATA_DIR / "clean_data.pkl")
df_full = df_full.rename(columns={'Selling_Price': 'target'})
Y = df_full['target']
X = df_full.drop('target', axis=1)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("Full sample:", X.shape)


In [ ]:
model_sklearn = pipeline_sklearn.fit(X, Y)

In [ ]:
predictions = model_sklearn.predict(X_test) 
metrics = {}
metrics["mae"] = mean_absolute_error(y_test, predictions)   
metrics["mape"] = mean_absolute_percentage_error(y_test, predictions)
metrics["mse"] = mean_squared_error(y_test, predictions)

metrics

In [ ]:
# Самая лучшая модель
experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
RUN_NAME = 'best_selection'
art = '../comment.txt'

# Список используемых столбцов
input_example = X.head(5)
signature = infer_signature(model_input=X.head(5))

cols_file = 'production_columns.txt'
with open(cols_file, 'w') as f:
    f.write('\n'.join(X.columns.tolist()))

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    mlflow.sklearn.log_model(
        pipeline_sklearn,
        artifact_path="models",
        signature=signature,
        input_example=input_example,
        pip_requirements=req_file,
        registered_model_name=REGISTRY_MODEL_NAME,
    )
    mlflow.log_artifact(art)
    mlflow.log_artifact(cols_file)
    mlflow.log_params(pipeline_sklearn.get_params())

run = mlflow.get_run(run_id)
assert run.info.status == 'FINISHED'

client = mlflow.tracking.MlflowClient()
versions = client.search_model_versions(f"name='{REGISTRY_MODEL_NAME}'")
latest = sorted(versions, key=lambda v: int(v.version))[-1]
client.set_registered_model_alias(REGISTRY_MODEL_NAME, 'Production', latest.version)
client.set_model_version_tag(REGISTRY_MODEL_NAME, latest.version, 'stage', 'Production')
print('Production version:', latest.version, 'run_id:', run_id)


In [ ]:
run